In [32]:
!pip install faiss-cpu
#!pip install faiss-gpu

In [1]:
from langchain_community.llms import Ollama

In [8]:
model = Ollama(model='gemma3:1b')

In [9]:
response = model.generate(["When did india gets independence??"])

In [15]:
response.generations[0][0].text

'India achieved independence on **August 15, 1947**. \n\nIt’s a momentous date! \n\n'

In [11]:
model_1 = Ollama(model='tinyllama:latest')

In [16]:
response_1 = model_1.generate(["When did india gets independence??"])
print(response_1.generations[0][0].text)

India got independence on August 15, 1947, which is celebrated as the Independence Day every year on this day. It was declared a Republic under the Constitution of India on October 26, 1949.


In [17]:
import openai
from langchain.vectorstores import FAISS
from langchain.document_loaders import PyPDFLoader
from langchain.embeddings import HuggingFaceEmbeddings
from langchain.text_splitter import RecursiveCharacterTextSplitter

In [19]:
pdf = PyPDFLoader(r'C:\Users\hp\Desktop\Tech Mango\Datasets\RAG.pdf')
doc = pdf.load()
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap = 200)
chunks = text_splitter.split_documents(doc)

In [33]:
Embedding = HuggingFaceEmbeddings()
db = FAISS.from_documents(documents = chunks, embedding = Embedding)

C:\Users\hp\AppData\Local\Temp\ipykernel_3284\1684145282.py:1: LangChainDeprecationWarning: Default values for HuggingFaceEmbeddings.model_name were deprecated in LangChain 0.2.16 and will be removed in 0.4.0. Explicitly pass a model_name to the HuggingFaceEmbeddings constructor instead.
  Embedding = HuggingFaceEmbeddings()


In [40]:
from langchain.chains import ConversationalRetrievalChain
from langchain.prompts import PromptTemplate

Question_Prompt = PromptTemplate.from_template('''Given the following Converstion
Chat history:
{chat_history}
Follow up question:
{question}
''')

qa = ConversationalRetrievalChain.from_llm(llm=model, retriever=db.as_retriever() ,condense_question_prompt=Question_Prompt,return_source_documents=True, verbose = False)

In [42]:
chat_history=[]
question = '''what is research paper all about?? explain in 5-6 lines'''
result = qa({"question": question, "chat_history": chat_history})
print("Answer:", result["answer"])
chat_history.append((question, result["answer"]))

Answer: The text describes the RAG (Retrieval-Augmented Generation) research paradigm, which focuses on enhancing information retrieval by combining document metadata with LLM-generated questions. It outlines three stages: Naive RAG, Iterative Retrieval, and Indexing Optimization. Naive RAG is a basic approach that searches based on the initial query, while Iterative Retrieval uses the generated text to refine the search. Indexing optimization involves creating vector embeddings to store the knowledge base. The quality of the index significantly impacts retrieval accuracy. Modular RAG is a response to limitations of Naive RAG, aiming for more sophisticated retrieval through layered techniques.


In [46]:
from transformers import GPT2LMHeadModel, GPT2Tokenizer
import torch

In [47]:
model_name = "gpt2"
model = GPT2LMHeadModel.from_pretrained(model_name)
tokenizer = GPT2Tokenizer.from_pretrained(model_name)
model.eval()

C:\Users\hp\.conda\envs\langchain_env\lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\hp\.cache\huggingface\hub\models--gpt2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better

GPT2LMHeadModel(
  (transformer): GPT2Model(
    (wte): Embedding(50257, 768)
    (wpe): Embedding(1024, 768)
    (drop): Dropout(p=0.1, inplace=False)
    (h): ModuleList(
      (0-11): 12 x GPT2Block(
        (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (attn): GPT2Attention(
          (c_attn): Conv1D(nf=2304, nx=768)
          (c_proj): Conv1D(nf=768, nx=768)
          (attn_dropout): Dropout(p=0.1, inplace=False)
          (resid_dropout): Dropout(p=0.1, inplace=False)
        )
        (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (mlp): GPT2MLP(
          (c_fc): Conv1D(nf=3072, nx=768)
          (c_proj): Conv1D(nf=768, nx=3072)
          (act): NewGELUActivation()
          (dropout): Dropout(p=0.1, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
  )
  (lm_head): Linear(in_features=768, out_features=50257, bias=False)
)

In [80]:
responses = [ "The capital of India is Delhi.",
             "India's capital is Delhi."
            ]

In [81]:
def calculate_ppl(text):
    inputs = tokenizer(text, return_tensors="pt")
    input_ids = inputs["input_ids"]

    with torch.no_grad():
        outputs = model(input_ids=input_ids, labels=input_ids)
        loss = outputs.loss
    
    ppl = torch.exp(loss).item()
    return loss.item(), ppl

for i, response in enumerate(responses, 1):
    loss, ppl_score = calculate_ppl(response)
    print(f"Response {i}: {response}")
    print(f"PPL Score: {ppl_score:.2f}\n")


Response 1: The capital of India is Delhi.
PPL Score: 46.50

Response 2: India's capital is Delhi.
PPL Score: 75.62

